# Flight Analytics — Event Hubs (Kafka protocol) Bronze Ingestion

Reads live flight data from Azure Event Hubs via the Kafka-compatible endpoint, and writes it untransformed into the Bronze Delta table.

**Prerequisites:**
- Event Hubs namespace on Standard tier or higher (Kafka protocol requires it)
- A Shared Access Policy with **Listen** claim (used here — NOT the Send/producer policy)
- Connection string stored in a Databricks secret scope (see note in the credentials cell)


In [ ]:
from pyspark.sql import functions as F

print("Spark version:", spark.version)


## Configuration

**Do not hardcode the connection string in a real/shared notebook.** Use a Databricks secret scope instead:
```python
EVENTHUB_CONNECTION_STRING = dbutils.secrets.get(scope="flight-analytics", key="eventhub-listen-conn-str")
```
The plain variable below is kept only for local/individual testing convenience.


In [ ]:
EVENTHUB_NAMESPACE = "flight-analytics-eh-anuj.servicebus.windows.net"
EVENTHUB_TOPIC = "flight_raw"

# IMPORTANT: this must be the connection string from the LISTEN policy
# (flight-spark-policy), not the producer's Send-only policy.
#
# Read from a Databricks secret scope - never hardcode this value.
# One-time setup (Databricks CLI): 
#   databricks secrets create-scope flight-analytics
#   databricks secrets put-secret flight-analytics eventhub-listen-conn-str
EVENTHUB_CONNECTION_STRING = dbutils.secrets.get(
    scope="flight-analytics", key="eventhub-listen-conn-str"
)

print("Namespace:", EVENTHUB_NAMESPACE)
print("Topic:", EVENTHUB_TOPIC)
print("Connection string loaded:", bool(EVENTHUB_CONNECTION_STRING))

## Kafka options for the Event Hubs Kafka-compatible surface

Event Hubs speaks the Kafka protocol over `SASL_SSL` with a `PLAIN` mechanism, where the username is the literal string `$ConnectionString` and the password is the actual connection string. This is Azure-specific — not how you'd configure a normal Kafka client.


In [ ]:
eventhub_kafka_options = {
    "kafka.bootstrap.servers": f"{EVENTHUB_NAMESPACE}:9093",
    "subscribe": EVENTHUB_TOPIC,
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": (
        "kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule "
        'required username="$ConnectionString" '
        f'password="{EVENTHUB_CONNECTION_STRING}";'
    ),
    # Azure Event Hubs recommended consumer settings
    "kafka.request.timeout.ms": "60000",
    "kafka.session.timeout.ms": "30000",
    # Event Hubs closes idle Kafka connections after 240 seconds server-side -
    # keep this comfortably under that.
    "kafka.connections.max.idle.ms": "180000",
    "kafka.metadata.max.age.ms": "180000",
    "failOnDataLoss": "false",
}

print("Kafka options configured for Event Hubs.")


## Create the streaming source


In [ ]:
raw_stream = (
    spark.readStream
    .format("kafka")
    .options(**eventhub_kafka_options)
    .option("startingOffsets", "earliest")
    .load()
)

print("Kafka streaming source created.")
print("Columns:", raw_stream.columns)


## Bronze write — append raw Kafka records to Delta, untransformed

This is the actual Bronze layer: minimal transformation (just casting `key`/`value` from bytes to string so they're human-readable), `outputMode("append")` since there's no aggregation, and a real checkpoint location so the query can resume correctly on restart.

`trigger(availableNow=True)` processes everything currently available then stops — useful for testing. Swap to `.trigger(processingTime="10 seconds")` for a truly continuously-running job.


In [ ]:
bronze_checkpoint = "/Volumes/flight_analytics/streaming/flight_data/checkpoints/bronze"
bronze_output_path = "/Volumes/flight_analytics/streaming/flight_data/bronze"

bronze_query = (
    raw_stream
    .select(
        F.col("key").cast("string").alias("key"),
        F.col("value").cast("string").alias("value"),
        "topic",
        "partition",
        "offset",
        "timestamp",
    )
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", bronze_checkpoint)
    .trigger(availableNow=True)
    .start(bronze_output_path)
)

bronze_query.awaitTermination()
print("Bronze write finished.")


## Quick sanity check — read back what landed in Bronze


In [ ]:
display(spark.read.format("delta").load(bronze_output_path).orderBy(F.col("timestamp").desc()).limit(20))
